# 🏥 NESA — Medical RAG System
## Notebook 01 · Document Ingestion, Extraction, Cleaning & Normalization

> **Scope:** This notebook handles only **ingestion → extraction → cleaning → normalization → saving**.
> Chunking, embeddings, vector databases, retrieval, and LLM generation are out of scope here.

---

| Stage | Description |
|---|---|
| **Input** | Raw PDF files under `data/raw/` |
| **Output** | Cleaned JSONL blocks under `data/processed/documents/` |
| **Traceability** | Every block retains `document_id`, `source_file`, `page`, `section` |


## 1. Configuration

> Edit the values below to match your environment.
> All paths are **relative** — no absolute paths are used.


In [ ]:
# ── Directory configuration ──────────────────────────────────────────────────
RAW_DIR       = "data/raw"
PROCESSED_DIR = "data/processed/documents"
TEXT_DIR      = "data/processed/text"          # human-readable .txt outputs

# ── Sub-directories inside data/raw/ ─────────────────────────────────────────
SUBDIR_GUIDELINES        = "guidelines"
SUBDIR_PATIENT_EDUCATION = "patient_education"
SUBDIR_RESEARCH_PAPERS   = "research_papers"

# ── Cleaning thresholds ───────────────────────────────────────────────────────
# If cleaned text is this % smaller than original, flag for manual review
REDUCTION_FLAG_THRESHOLD = 0.40   # 40 % reduction triggers a warning

# How many consecutive pages must share identical text for it to be
# considered a header/footer artifact
HEADER_FOOTER_MIN_PAGES = 3

# ── Output files ─────────────────────────────────────────────────────────────
JSONL_OUTPUT  = f"{PROCESSED_DIR}/nesa_documents.jsonl"
REPORT_FILE   = f"{PROCESSED_DIR}/cleaning_report.json"
FAILURE_FILE  = f"{PROCESSED_DIR}/failure_report.json"

print("✅  Configuration loaded.")
print(f"   RAW_DIR       → {RAW_DIR}")
print(f"   PROCESSED_DIR → {PROCESSED_DIR}")
print(f"   JSONL_OUTPUT  → {JSONL_OUTPUT}")


## 2. Imports

In [ ]:
import os
import re
import json
import unicodedata
import hashlib
import warnings
import logging
from pathlib import Path
from collections import Counter, defaultdict
from typing import Optional, List, Dict, Any, Tuple

import fitz                     # PyMuPDF  — primary PDF extraction engine
import pdfplumber                # secondary engine (better table detection)
import pandas as pd
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.WARNING)

# ── Create output directories ─────────────────────────────────────────────────
Path(PROCESSED_DIR).mkdir(parents=True, exist_ok=True)
Path(TEXT_DIR).mkdir(parents=True, exist_ok=True)

print("✅  All imports successful.")


## 3. Input Directory Discovery

**Why this step?**
We recursively discover all PDF files and assign a deterministic `document_id` based on the
file path hash. This guarantees stable identifiers across pipeline re-runs.


In [ ]:
def discover_documents(raw_dir: str) -> List[Dict[str, str]]:
    """
    Recursively find all PDF files under raw_dir.
    Returns a list of dicts with: path, document_id, document_type.
    """
    raw_path = Path(raw_dir)
    documents = []

    # Map sub-directory names → document_type labels
    TYPE_MAP = {
        "guidelines":        "guideline",
        "patient_education": "patient_education",
        "research_papers":   "research_paper",
    }

    for pdf_path in sorted(raw_path.rglob("*.pdf")):
        # Infer document type from parent directory name
        parent = pdf_path.parent.name
        doc_type = TYPE_MAP.get(parent, "book")  # default to 'book'

        # Stable document_id from relative path hash
        rel = pdf_path.relative_to(raw_path)
        doc_id = hashlib.md5(str(rel).encode()).hexdigest()[:12]

        documents.append({
            "document_id":   doc_id,
            "source_file":   str(pdf_path),
            "document_type": doc_type,
            "filename":      pdf_path.name,
        })

    return documents


# ── Run discovery ─────────────────────────────────────────────────────────────
# Create sample directories if they don't exist (for demo purposes)
for subdir in [SUBDIR_GUIDELINES, SUBDIR_PATIENT_EDUCATION, SUBDIR_RESEARCH_PAPERS]:
    Path(RAW_DIR, subdir).mkdir(parents=True, exist_ok=True)

discovered = discover_documents(RAW_DIR)
print(f"📂  Documents found: {len(discovered)}")
for doc in discovered:
    print(f"   [{doc['document_type']:18s}]  {doc['filename']}")

if not discovered:
    print("\n⚠️  No PDFs found. Place PDF files in:")
    print(f"      {RAW_DIR}/guidelines/")
    print(f"      {RAW_DIR}/patient_education/")
    print(f"      {RAW_DIR}/research_papers/")


## 4. PDF Extraction

**Why two engines?**
PyMuPDF (`fitz`) is fast and reliable for text extraction; `pdfplumber` is more accurate for
table detection. We use both and merge the results.

**Limitations of PDF extraction:**
- Scanned PDFs (images only) produce no extractable text without OCR.
- Multi-column layouts may mix column text.
- Mathematical formulae are often garbled.
- Some PDFs use non-standard encoding.

These cases are flagged for manual review.


In [ ]:
def extract_pdf(source_file: str) -> Dict[str, Any]:
    """
    Extract text page-by-page using PyMuPDF (primary) and pdfplumber (for tables).
    Returns a dict with: pages (list), metadata (dict), tables (list), errors (list).
    """
    result = {
        "pages":    [],   # list of {page_num, text, blocks}
        "tables":   [],   # raw table data extracted by pdfplumber
        "metadata": {},
        "errors":   [],
    }

    # ── PyMuPDF extraction ────────────────────────────────────────────────────
    try:
        doc_mu = fitz.open(source_file)

        # Metadata
        meta = doc_mu.metadata or {}
        result["metadata"] = {
            "title":            meta.get("title")  or None,
            "author":           meta.get("author") or None,
            "subject":          meta.get("subject") or None,
            "creator":          meta.get("creator") or None,
            "page_count":       doc_mu.page_count,
            "pdf_version":      doc_mu.pdf_version() if hasattr(doc_mu, "pdf_version") else None,
        }

        for page_num, page in enumerate(doc_mu, start=1):
            # Extract text preserving reading order
            text = page.get_text("text", sort=True)

            # Extract structured text blocks (for heading detection)
            blocks_raw = page.get_text("blocks", sort=True)
            blocks = []
            for b in blocks_raw:
                # b = (x0, y0, x1, y1, text, block_no, block_type)
                if len(b) >= 6 and b[6] == 0:   # block_type 0 = text
                    blocks.append({
                        "text":   b[4].strip(),
                        "bbox":   b[:4],
                        "block_no": b[5],
                    })

            result["pages"].append({
                "page_num": page_num,
                "text":     text,
                "blocks":   blocks,
            })

        doc_mu.close()

    except Exception as e:
        result["errors"].append(f"PyMuPDF error: {e}")

    # ── pdfplumber table extraction ───────────────────────────────────────────
    try:
        with pdfplumber.open(source_file) as doc_pl:
            for page_num, page in enumerate(doc_pl.pages, start=1):
                tables = page.extract_tables()
                for tbl in tables:
                    if tbl:
                        result["tables"].append({
                            "page_num": page_num,
                            "raw":      tbl,
                        })
    except Exception as e:
        result["errors"].append(f"pdfplumber error: {e}")

    return result

print("✅  extract_pdf() defined.")


## 5. Metadata Extraction

We extract metadata from PDF properties and, when available, from the document text
(e.g., year from abstract, author from title page). We **never fabricate** missing values.


In [ ]:
# Common year pattern e.g. (2023) or ©2021 or Published 2019
YEAR_RE = re.compile(r'\b(19[5-9]\d|20[0-2]\d)\b')

def extract_metadata(doc_info: Dict, raw_extraction: Dict, source_file: str) -> Dict:
    """
    Combine PDF metadata with heuristic extraction.
    Never fabricates information — uses None for missing fields.
    """
    pdf_meta = raw_extraction.get("metadata", {})

    # Try to find a publication year in the first 2 pages
    year = None
    first_pages_text = " ".join(
        p["text"] for p in raw_extraction["pages"][:2]
    )
    year_matches = YEAR_RE.findall(first_pages_text)
    if year_matches:
        # Take the most common year in the first pages
        year = int(Counter(year_matches).most_common(1)[0][0])

    # Author: try PDF metadata first, then None
    raw_author = pdf_meta.get("author") or None
    authors = None
    if raw_author:
        # Split on common separators
        authors = [a.strip() for a in re.split(r'[,;&]|\band\b', raw_author) if a.strip()]

    title = pdf_meta.get("title") or None
    # If title looks like a filename artifact, discard it
    if title and (len(title) < 3 or title.lower().endswith(".pdf")):
        title = None

    return {
        "title":            title,
        "authors":          authors,
        "publication_year": year,
        "page_count":       pdf_meta.get("page_count"),
        "source_file":      source_file,
        "document_type":    doc_info["document_type"],
        "document_id":      doc_info["document_id"],
        "filename":         doc_info["filename"],
    }

print("✅  extract_metadata() defined.")


## 6. Header / Footer Detection

**Why this step?**
Repeated text at the top/bottom of pages (running headers, footers, journal names,
website URLs) pollutes the RAG corpus. We detect them by frequency analysis across pages.

**Safety rule:** We only remove a candidate if it appears on ≥ `HEADER_FOOTER_MIN_PAGES`
consecutive pages, to avoid deleting legitimate short section headings.


In [ ]:
def detect_headers_footers(pages: List[Dict], min_pages: int = HEADER_FOOTER_MIN_PAGES) -> set:
    """
    Identify repeated strings that appear at the very top or bottom of many pages.
    Returns a set of normalized strings to treat as artifacts.
    """
    # Collect first and last non-empty line of each page
    top_lines    = []
    bottom_lines = []

    for page in pages:
        lines = [l.strip() for l in page["text"].split("\n") if l.strip()]
        if lines:
            top_lines.append(lines[0])
            bottom_lines.append(lines[-1])

    artifacts = set()

    for candidates in (top_lines, bottom_lines):
        freq = Counter(candidates)
        for text, count in freq.items():
            if count >= min_pages and len(text) > 0:
                # Extra guard: must not look like a dosage or medical value
                if not re.search(r'\d+\s*(mg|ml|g|kg|%|ci|p\s*[<=>])', text, re.I):
                    artifacts.add(text)

    return artifacts

print("✅  detect_headers_footers() defined.")


## 7. Page-Number Removal

**Why this step?**
Standalone page numbers extracted by PDF parsers appear as isolated tokens that add noise.

**Safety rule:** We only remove a line when it is *clearly* a page-number artifact:
- A lone integer.
- "Page N", "N | Chapter X", etc.

We **never** remove numbers embedded in medical content such as dosages or statistics.


In [ ]:
# Patterns that indicate a standalone page-number artifact
PAGE_NUM_PATTERNS = [
    re.compile(r'^\d{1,4}$'),                          # "42"
    re.compile(r'^[Pp]age\s+\d{1,4}$'),               # "Page 42"
    re.compile(r'^\d{1,4}\s*\|.*$'),                 # "42 | Chapter 3"
    re.compile(r'^-\s*\d{1,4}\s*-$'),                # "- 42 -"
    re.compile(r'^\[\s*\d{1,4}\s*\]$'),            # "[42]"
]

# Patterns that must NEVER be removed (medical content)
MEDICAL_SAFE_PATTERNS = [
    re.compile(r'\d+\s*(mg|ml|g|kg|mcg|µg|IU|mEq)', re.I),
    re.compile(r'\d+\s*(days?|weeks?|months?|hours?|minutes?)', re.I),
    re.compile(r'\b(table|figure|fig|appendix|equation)\s*\d+', re.I),
    re.compile(r'\d{4}'),                              # years
    re.compile(r'\d+\s*%'),
    re.compile(r'p\s*[<=>]\s*0\.\d+', re.I),
    re.compile(r'\d+(\.\d+)?\s*(CI|OR|RR|HR)', re.I),
]


def is_page_number_artifact(line: str) -> bool:
    """Return True only if the line is clearly a page-number artifact."""
    line = line.strip()
    if not line:
        return False

    # Safety first: if it matches medical content, keep it
    for pattern in MEDICAL_SAFE_PATTERNS:
        if pattern.search(line):
            return False

    # Now check page-number patterns
    for pattern in PAGE_NUM_PATTERNS:
        if pattern.match(line):
            return True

    return False


def remove_page_numbers(text: str) -> Tuple[str, int]:
    """
    Remove page-number artifacts from text.
    Returns (cleaned_text, count_removed).
    """
    lines = text.split("\n")
    cleaned = []
    removed = 0
    for line in lines:
        if is_page_number_artifact(line):
            removed += 1
        else:
            cleaned.append(line)
    return "\n".join(cleaned), removed

print("✅  remove_page_numbers() defined.")


## 8. Text Normalization

**Steps applied:**
1. Unicode normalization (NFC) — resolves ligatures and composed characters.
2. Replace non-breaking spaces, zero-width characters.
3. Normalize multiple spaces → single space.
4. Normalize line endings.
5. Remove excessive blank lines (> 2 consecutive).
6. Strip binary/control characters.

**Preserved:** All meaningful medical characters including special symbols (±, ≥, ≤, μ, α, β).


In [ ]:
# Characters that are meaningful in medical text and must be preserved
PRESERVE_CHARS = set('±≥≤μαβγδσΩ°→←↑↓∞≠≈×÷√∑∏∫%§†‡')

def clean_text(text: str) -> str:
    """
    Apply deterministic Unicode and whitespace normalization.
    Does NOT remove medical content.
    """
    # 1. Unicode NFC normalization
    text = unicodedata.normalize("NFC", text)

    # 2. Replace common problematic characters
    replacements = {
        '\u00a0': ' ',    # non-breaking space
        '\u200b': '',     # zero-width space
        '\u200c': '',     # zero-width non-joiner
        '\u200d': '',     # zero-width joiner
        '\ufeff': '',     # BOM
        '\u2028': '\n',  # line separator
        '\u2029': '\n',  # paragraph separator
        '\u2013': '–',    # en-dash
        '\u2014': '—',    # em-dash
        '\u2018': "'",    # left single quotation
        '\u2019': "'",    # right single quotation
        '\u201c': '"',    # left double quotation
        '\u201d': '"',    # right double quotation
        '\u2022': '•',    # bullet
        '\u00b7': '·',    # middle dot
    }
    for old, new in replacements.items():
        text = text.replace(old, new)

    # 3. Remove control characters (keep newline, tab)
    text = ''.join(
        ch for ch in text
        if unicodedata.category(ch)[0] != 'C'
        or ch in ('\n', '\t', '\r')
        or ch in PRESERVE_CHARS
    )

    # 4. Normalize whitespace within lines (multiple spaces → one)
    lines = text.split("\n")
    lines = [re.sub(r'[ \t]+', ' ', line).strip() for line in lines]

    # 5. Normalize multiple blank lines → max 2
    cleaned_lines = []
    blank_count = 0
    for line in lines:
        if line == '':
            blank_count += 1
            if blank_count <= 2:
                cleaned_lines.append(line)
        else:
            blank_count = 0
            cleaned_lines.append(line)

    return "\n".join(cleaned_lines)

print("✅  clean_text() defined.")


## 9. Hyphenation Repair

**Problem:**
PDF line-wrapping often breaks words across lines with a hyphen:
```
breast-
feeding
```

**Solution:**
Detect hyphen at end of line followed by continuation on next line. Join them.

**Safety rule:**
Preserve legitimate compound hyphenated terms like `evidence-based`, `follow-up`,
`post-operative`, etc. These are recognized because they appear entirely on one line.


In [ ]:
# Known legitimate medical hyphenated terms — never join these across lines
LEGITIMATE_HYPHENATED = {
    'evidence-based', 'follow-up', 'post-operative', 'pre-eclampsia',
    'skin-to-skin', 'well-being', 'long-term', 'short-term',
    'double-blind', 'placebo-controlled', 'dose-dependent',
    'cross-sectional', 'case-control', 'risk-benefit',
    'non-pharmacological', 'non-invasive', 'breast-feeding',
}


def repair_hyphenation(text: str) -> Tuple[str, int]:
    """
    Join words broken by PDF line-wrap hyphenation.
    Returns (repaired_text, count_repairs).
    """
    lines = text.split("\n")
    result = []
    repairs = 0
    i = 0

    while i < len(lines):
        line = lines[i]

        # Check if line ends with a hyphen and next line starts with a lowercase letter
        if (i + 1 < len(lines)
                and line.endswith('-')
                and lines[i + 1]
                and lines[i + 1][0].islower()):

            # Candidate joined word
            next_line = lines[i + 1]
            joined = line[:-1] + next_line.split()[0] if next_line.split() else line

            # Only join if the joined word is NOT a known legitimate hyphenated term
            if joined.lower() not in LEGITIMATE_HYPHENATED:
                # Merge: remove hyphen, join with rest of next line
                rest_of_next = ' '.join(next_line.split()[1:])
                merged = line[:-1] + next_line.split()[0] if next_line.split() else line[:-1]
                if rest_of_next:
                    merged += ' ' + rest_of_next
                result.append(merged)
                repairs += 1
                i += 2
                continue

        result.append(line)
        i += 1

    return "\n".join(result), repairs

print("✅  repair_hyphenation() defined.")


## 10. Paragraph Reconstruction

**Problem:**
PDF extraction often produces one-sentence-per-line output:
```
The patient should remain hydrated
during the postpartum period and
should seek medical attention if...
```

**Solution:**
Merge lines that are clearly continuations of the same paragraph (no sentence-ending
punctuation, line is not a heading, not a list item).

**Preserved:** Real paragraph breaks (blank lines), headings, list items.


In [ ]:
# Patterns that indicate a new paragraph / heading should NOT be merged
NEW_PARA_PATTERNS = [
    re.compile(r'^#+\s'),                             # markdown heading
    re.compile(r'^\d+\.\s'),                        # numbered list
    re.compile(r'^[•\-\*]\s'),                      # bullet list
    re.compile(r'^(abstract|introduction|methods?|results?|discussion|conclusion|references?|appendix|chapter|section)\b', re.I),
    re.compile(r'^(table|figure|fig\.?)\s*\d+', re.I),
    re.compile(r'^[A-Z][A-Z\s]{5,}$'),               # ALL CAPS heading
]

SENTENCE_END = re.compile(r'[.!?]\s*$')


def _is_new_paragraph_start(line: str) -> bool:
    """Return True if line should start a new paragraph."""
    if not line:
        return True
    for p in NEW_PARA_PATTERNS:
        if p.match(line.strip()):
            return True
    # Very short lines followed by a capital probably are headings
    if len(line.strip()) < 50 and line.strip() and line.strip()[0].isupper():
        words = line.strip().split()
        if len(words) <= 5:
            return True
    return False


def normalize_paragraphs(text: str) -> str:
    """
    Merge soft-wrapped lines back into proper paragraphs.
    Preserves real paragraph breaks (blank lines) and headings.
    """
    lines = text.split("\n")
    paragraphs = []
    current = []

    for line in lines:
        stripped = line.strip()

        if stripped == '':
            # Blank line → flush current paragraph
            if current:
                paragraphs.append(' '.join(current))
                current = []
            paragraphs.append('')
            continue

        if _is_new_paragraph_start(stripped):
            if current:
                paragraphs.append(' '.join(current))
                current = []
            paragraphs.append(stripped)
            continue

        # If previous line ended with sentence-ending punctuation, start new paragraph
        if current and SENTENCE_END.search(current[-1]):
            paragraphs.append(' '.join(current))
            current = [stripped]
        else:
            current.append(stripped)

    if current:
        paragraphs.append(' '.join(current))

    # Collapse multiple blank lines
    result = []
    prev_blank = False
    for p in paragraphs:
        if p == '':
            if not prev_blank:
                result.append('')
            prev_blank = True
        else:
            result.append(p)
            prev_blank = False

    return "\n".join(result)

print("✅  normalize_paragraphs() defined.")


## 11. Heading Detection

**Why preserve headings?**
Headings provide retrieval context. A chunk about "500 mg" means more when the retriever
knows it lives under "Postpartum Medication Guidelines > Pain Management".

We detect headings by:
- ALL CAPS short lines.
- Known section keywords (Abstract, Introduction, Methods…).
- PyMuPDF font-size blocks (larger font = likely heading).


In [ ]:
SECTION_KEYWORDS = {
    'abstract', 'introduction', 'background', 'methods', 'methodology',
    'results', 'findings', 'discussion', 'conclusion', 'conclusions',
    'recommendations', 'references', 'bibliography', 'appendix',
    'acknowledgements', 'acknowledgments', 'supplementary',
    'postpartum', 'breastfeeding', 'nutrition', 'recovery',
    'chapter', 'section', 'overview', 'summary', 'objectives',
}


def detect_heading(line: str) -> Optional[str]:
    """
    Return heading level ('h1', 'h2', 'h3') or None.
    h1 → ALL CAPS / known chapter-level keywords
    h2 → Title Case section keywords
    h3 → Bold/short mixed-case lines
    """
    s = line.strip()
    if not s or len(s) > 120:
        return None

    # ALL CAPS (more than 3 words or starts with CHAPTER/SECTION)
    if s == s.upper() and len(s.split()) >= 2 and re.search(r'[A-Z]', s):
        return 'h1'

    # Starts with "Chapter N" / "Section N"
    if re.match(r'^(Chapter|Section|Part)\s+\d+', s, re.I):
        return 'h1'

    # Known section keyword at start of line
    first_word = s.split()[0].lower().rstrip(':.,')
    if first_word in SECTION_KEYWORDS and len(s.split()) <= 6:
        return 'h2'

    # Title Case and short (likely a subsection heading)
    words = s.split()
    if (len(words) <= 8
            and all(w[0].isupper() for w in words if len(w) > 3)
            and not s.endswith('.')):
        return 'h3'

    return None


def format_heading(line: str, level: str) -> str:
    """Format a heading with markdown-style prefix."""
    prefix = {'h1': '# ', 'h2': '## ', 'h3': '### '}.get(level, '')
    return prefix + line.strip()

print("✅  detect_heading() and format_heading() defined.")


## 12. Image and Figure Handling

**Policy:**
- Image binary data is NOT inserted into the RAG corpus.
- Figure captions containing clinical information are preserved as text.
- Image placeholders / OCR artifacts are removed.

**Preserved:**
```
[Figure: Recommended breastfeeding position]
```


In [ ]:
FIGURE_PATTERNS = [
    re.compile(r'^(Figure|Fig\.?)\s*\d+[.:]?\s*(.+)', re.I),
    re.compile(r'^(Plate|Image|Illustration)\s*\d+[.:]?\s*(.+)', re.I),
]

IMAGE_ARTIFACT_PATTERNS = [
    re.compile(r'^\s*<image>\s*$', re.I),
    re.compile(r'^\s*\[image\]\s*$', re.I),
    re.compile(r'^\s*\[fig(ure)?\]\s*$', re.I),
    re.compile(r'^\ufffd+$'),                # Unicode replacement characters
    re.compile(r'^[\x00-\x08\x0b-\x1f]+$'),  # Binary artifacts
]


def process_figures(text: str) -> Tuple[str, List[str]]:
    """
    Detect figure captions → convert to [Figure: ...] format.
    Remove image artifacts.
    Returns (cleaned_text, list_of_figure_captions).
    """
    lines = text.split("\n")
    result = []
    figure_captions = []

    for line in lines:
        stripped = line.strip()

        # Remove image artifacts
        is_artifact = any(p.match(stripped) for p in IMAGE_ARTIFACT_PATTERNS)
        if is_artifact:
            continue

        # Detect figure captions
        is_figure = False
        for p in FIGURE_PATTERNS:
            m = p.match(stripped)
            if m:
                caption = m.group(2).strip() if len(m.groups()) >= 2 else stripped
                formatted = f"[Figure: {caption}]"
                result.append(formatted)
                figure_captions.append(formatted)
                is_figure = True
                break

        if not is_figure:
            result.append(line)

    return "\n".join(result), figure_captions

print("✅  process_figures() defined.")


## 13. Table Extraction and Conversion

**Critical:** Tables contain medically important information (dosages, trial results,
vital signs thresholds). They must **never** be silently deleted.

**Strategy:**
1. Use pdfplumber's table data (list of rows).
2. Convert to a structured text representation preserving row-column relationships.
3. If extraction is unreliable, output a fallback and flag for review.

**Example conversion:**
```
TABLE: Medication Dosing
- Drug A: dose = 500 mg; frequency = twice daily.
- Drug B: dose = 250 mg; frequency = once daily.
```


In [ ]:
def table_to_text(raw_table: List[List], page_num: int, table_index: int = 0) -> Dict[str, Any]:
    """
    Convert a pdfplumber raw table (list of rows) to structured text.
    Returns a dict suitable for a JSONL record.
    """
    if not raw_table:
        return {
            "content_type":  "table",
            "text":          "[TABLE: empty or unreadable]",
            "table_title":   None,
            "cleaning_flags": ["empty_table"],
        }

    # Clean cells: replace None with empty string
    cleaned = []
    for row in raw_table:
        if row:
            cleaned.append([str(cell).strip() if cell is not None else '' for cell in row])

    if not cleaned:
        return {
            "content_type":  "table",
            "text":          "[TABLE: could not parse]",
            "table_title":   None,
            "cleaning_flags": ["parse_failed"],
        }

    # First row → headers (heuristic: if first row has no numerics)
    headers = cleaned[0]
    data_rows = cleaned[1:]

    # Build textual representation
    lines = [f"TABLE (page {page_num}):"]

    # Column headers
    header_str = " | ".join(h for h in headers if h)
    if header_str:
        lines.append(f"Columns: {header_str}")

    # Rows
    for row in data_rows:
        if not any(cell for cell in row):
            continue   # skip completely empty rows

        # Map header → value
        if len(headers) == len(row):
            parts = [f"{h} = {v}" for h, v in zip(headers, row) if h or v]
        else:
            parts = [v for v in row if v]

        if parts:
            lines.append("- " + "; ".join(parts) + ".")

    full_text = "\n".join(lines)

    # Flag if table has very few cells (might be a false positive)
    flags = []
    total_cells = sum(len(row) for row in cleaned)
    if total_cells < 4:
        flags.append("low_cell_count_review")

    return {
        "content_type":  "table",
        "text":          full_text,
        "table_title":   None,
        "cleaning_flags": flags,
    }


def extract_tables(raw_extraction: Dict) -> List[Dict]:
    """
    Convert all extracted raw tables to text records.
    """
    table_records = []
    for idx, tbl in enumerate(raw_extraction.get("tables", [])):
        record = table_to_text(tbl["raw"], tbl["page_num"], idx)
        record["page_num"] = tbl["page_num"]
        table_records.append(record)
    return table_records

print("✅  table_to_text() and extract_tables() defined.")


## 14. References Detection

References are **not** deleted — they are separated into their own section.
This supports citation tracing for the Research Assistant use case.


In [ ]:
REFERENCE_START = re.compile(
    r'^(References?|Bibliography|Works\s+Cited|Literature\s+Cited)',
    re.I
)

def split_references(text: str) -> Tuple[str, str]:
    """
    Split text into (main_body, references_section).
    If no references section is found, returns (text, '').
    """
    lines = text.split("\n")
    ref_start_idx = None

    for i, line in enumerate(lines):
        if REFERENCE_START.match(line.strip()):
            ref_start_idx = i
            break

    if ref_start_idx is None:
        return text, ''

    main   = "\n".join(lines[:ref_start_idx])
    refs   = "\n".join(lines[ref_start_idx:])
    return main, refs

print("✅  split_references() defined.")


## 15. Document Block Creation

Each logical content block (paragraph, heading, table, figure caption, reference) becomes
one JSONL record. Every record carries full traceability metadata.


In [ ]:
def create_document_blocks(
    cleaned_pages: List[Dict],
    table_records:  List[Dict],
    metadata:       Dict,
    header_footer_artifacts: set,
) -> List[Dict]:
    """
    Create a flat list of content blocks from cleaned page text and tables.
    Each block is one JSONL record.
    """
    blocks = []
    current_section = None
    current_chapter = None
    block_index = 0

    def make_block(text, content_type, page_num, flags=None):
        nonlocal block_index
        b = {
            "document_id":      metadata["document_id"],
            "source_file":      metadata["source_file"],
            "document_type":    metadata["document_type"],
            "title":            metadata["title"],
            "authors":          metadata["authors"],
            "publication_year": metadata["publication_year"],
            "page":             page_num,
            "chapter":          current_chapter,
            "section":          current_section,
            "content_type":     content_type,
            "text":             text.strip(),
            "cleaning_flags":   flags or [],
            "block_index":      block_index,
        }
        block_index += 1
        return b

    # ── Process page text ─────────────────────────────────────────────────────
    for page_data in cleaned_pages:
        page_num = page_data["page_num"]
        text     = page_data["cleaned_text"]
        _, refs  = split_references(text)

        paragraphs = text.split("\n")

        for para in paragraphs:
            para = para.strip()
            if not para:
                continue

            # Skip header/footer artifacts
            if para in header_footer_artifacts:
                continue

            # Detect heading
            level = detect_heading(para)
            if level:
                formatted = format_heading(para, level)
                if level == 'h1':
                    current_chapter = para
                    current_section = None
                elif level in ('h2', 'h3'):
                    current_section = para
                blocks.append(make_block(formatted, "heading", page_num))
                continue

            # Detect figure caption
            if para.startswith('[Figure:'):
                blocks.append(make_block(para, "figure_caption", page_num))
                continue

            # Detect reference line
            if refs and para in refs:
                blocks.append(make_block(para, "reference", page_num))
                continue

            # Regular paragraph
            if len(para) > 20:   # skip very short noisy fragments
                blocks.append(make_block(para, "paragraph", page_num))

    # ── Inject table blocks ───────────────────────────────────────────────────
    for tbl in table_records:
        b = make_block(tbl["text"], "table", tbl["page_num"], tbl["cleaning_flags"])
        b["table_title"] = tbl.get("table_title")
        blocks.append(b)

    # Sort by page then block_index
    blocks.sort(key=lambda x: (x["page"] or 0, x["block_index"]))
    return blocks

print("✅  create_document_blocks() defined.")


## 16. Quality Checks & Validation


In [ ]:
def validate_document(blocks: List[Dict], original_char_count: int) -> Dict:
    """
    Compute quality metrics for one document.
    Returns a report dict.
    """
    total_text = " ".join(b["text"] for b in blocks)
    cleaned_chars = len(total_text)

    reduction = 1.0 - (cleaned_chars / original_char_count) if original_char_count > 0 else 0.0

    type_counts = Counter(b["content_type"] for b in blocks)
    all_flags   = [f for b in blocks for f in b.get("cleaning_flags", [])]

    return {
        "original_chars":        original_char_count,
        "cleaned_chars":         cleaned_chars,
        "reduction_pct":         round(reduction * 100, 1),
        "needs_review":          reduction > REDUCTION_FLAG_THRESHOLD,
        "block_count":           len(blocks),
        "paragraphs":            type_counts.get("paragraph", 0),
        "headings":              type_counts.get("heading", 0),
        "tables":                type_counts.get("table", 0),
        "figure_captions":       type_counts.get("figure_caption", 0),
        "references":            type_counts.get("reference", 0),
        "warnings":              all_flags,
        "warning_count":         len(all_flags),
    }

print("✅  validate_document() defined.")


## 17. Save Cleaned Documents


In [ ]:
def save_jsonl(blocks: List[Dict], output_path: str, mode: str = 'a'):
    """Append blocks to a JSONL file."""
    with open(output_path, mode, encoding='utf-8') as f:
        for block in blocks:
            f.write(json.dumps(block, ensure_ascii=False) + "\n")


def save_text(pages: List[Dict], output_dir: str, document_id: str):
    """Save human-readable cleaned text per document."""
    out_path = Path(output_dir) / f"{document_id}.txt"
    with open(out_path, 'w', encoding='utf-8') as f:
        for page_data in pages:
            f.write(f"\n{'='*60}\n")
            f.write(f"PAGE {page_data['page_num']}\n")
            f.write(f"{'='*60}\n")
            f.write(page_data['cleaned_text'])
            f.write("\n")

print("✅  save_jsonl() and save_text() defined.")


## 18. Main Processing Pipeline

This cell orchestrates all steps for every discovered PDF.
Failed documents are caught and logged without stopping the pipeline.


In [ ]:
def process_single_document(doc_info: Dict) -> Tuple[Optional[List[Dict]], Dict, Optional[str]]:
    """
    Full pipeline for one PDF.
    Returns (blocks, report, error_message).
    """
    source_file = doc_info["source_file"]

    try:
        # ── Step 1: Extract ──────────────────────────────────────────────────
        raw = extract_pdf(source_file)
        if raw["errors"] and not raw["pages"]:
            return None, {}, f"Extraction failed: {raw['errors']}"

        # ── Step 2: Metadata ────────────────────────────────────────────────
        metadata = extract_metadata(doc_info, raw, source_file)

        # ── Step 3: Detect headers/footers ──────────────────────────────────
        artifacts = detect_headers_footers(raw["pages"])

        # ── Step 4: Process each page ───────────────────────────────────────
        original_chars = 0
        cleaned_pages  = []
        total_pn_removed   = 0
        total_repairs      = 0

        for page_data in raw["pages"]:
            raw_text = page_data["text"]
            original_chars += len(raw_text)

            # Remove header/footer artifacts from text
            lines = raw_text.split("\n")
            lines = [l for l in lines if l.strip() not in artifacts]
            text = "\n".join(lines)

            # Page-number removal
            text, pn_removed = remove_page_numbers(text)
            total_pn_removed += pn_removed

            # Unicode + whitespace normalization
            text = clean_text(text)

            # Hyphenation repair
            text, repairs = repair_hyphenation(text)
            total_repairs += repairs

            # Figure handling
            text, _ = process_figures(text)

            # Paragraph reconstruction
            text = normalize_paragraphs(text)

            cleaned_pages.append({
                "page_num":    page_data["page_num"],
                "cleaned_text": text,
            })

        # ── Step 5: Table extraction ────────────────────────────────────────
        table_records = extract_tables(raw)

        # ── Step 6: Build blocks ────────────────────────────────────────────
        blocks = create_document_blocks(cleaned_pages, table_records, metadata, artifacts)

        # ── Step 7: Validate ────────────────────────────────────────────────
        report = validate_document(blocks, original_chars)
        report["document_id"]      = metadata["document_id"]
        report["source_file"]      = source_file
        report["page_count"]       = len(cleaned_pages)
        report["pn_removed"]       = total_pn_removed
        report["hyphen_repairs"]   = total_repairs
        report["header_footers"]   = len(artifacts)
        report["extraction_errors"] = raw["errors"]

        # ── Step 8: Save text ───────────────────────────────────────────────
        save_text(cleaned_pages, TEXT_DIR, metadata["document_id"])

        return blocks, report, None

    except Exception as e:
        import traceback
        return None, {}, traceback.format_exc()


# ── Clear previous JSONL output ───────────────────────────────────────────────
if Path(JSONL_OUTPUT).exists():
    Path(JSONL_OUTPUT).unlink()

# ── Run pipeline ──────────────────────────────────────────────────────────────
cleaning_reports = []
failure_reports  = []
all_block_counts = []

print(f"🚀  Processing {len(discovered)} document(s)...\n")

for doc_info in tqdm(discovered, desc="Processing PDFs"):
    blocks, report, error = process_single_document(doc_info)

    if error:
        failure_reports.append({
            "document_id": doc_info["document_id"],
            "source_file": doc_info["source_file"],
            "error":       error,
        })
        print(f"  ❌  FAILED: {doc_info['filename']}")
        print(f"      {error[:200]}")
    else:
        save_jsonl(blocks, JSONL_OUTPUT)
        cleaning_reports.append(report)
        all_block_counts.append(len(blocks))
        status = "⚠️  NEEDS REVIEW" if report.get("needs_review") else "✅"
        print(f"  {status}  {doc_info['filename']}  "
              f"({report['block_count']} blocks, "
              f"{report['reduction_pct']}% reduction)")

print(f"\n✅  Done. {len(cleaning_reports)} succeeded, {len(failure_reports)} failed.")


## 19. Cleaning Report


In [ ]:
# ── Aggregate statistics ─────────────────────────────────────────────────────
total_pdfs      = len(discovered)
total_success   = len(cleaning_reports)
total_failed    = len(failure_reports)
total_pages     = sum(r.get("page_count", 0)        for r in cleaning_reports)
total_blocks    = sum(r.get("block_count", 0)       for r in cleaning_reports)
total_tables    = sum(r.get("tables", 0)            for r in cleaning_reports)
total_figs      = sum(r.get("figure_captions", 0)  for r in cleaning_reports)
total_hf        = sum(r.get("header_footers", 0)   for r in cleaning_reports)
total_pn        = sum(r.get("pn_removed", 0)       for r in cleaning_reports)
total_hyphen    = sum(r.get("hyphen_repairs", 0)   for r in cleaning_reports)
total_warnings  = sum(r.get("warning_count", 0)    for r in cleaning_reports)
needs_review    = [r for r in cleaning_reports if r.get("needs_review")]

print("=" * 60)
print("NESA — CLEANING REPORT")
print("=" * 60)
print(f"  PDFs discovered          : {total_pdfs}")
print(f"  Successfully processed   : {total_success}")
print(f"  Failed                   : {total_failed}")
print(f"  Pages processed          : {total_pages}")
print(f"  Content blocks created   : {total_blocks}")
print(f"  Tables extracted         : {total_tables}")
print(f"  Figure captions          : {total_figs}")
print(f"  Header/footer artifacts  : {total_hf}")
print(f"  Page numbers removed     : {total_pn}")
print(f"  Hyphenation repairs      : {total_hyphen}")
print(f"  Cleaning warnings        : {total_warnings}")
print(f"  Docs needing review      : {len(needs_review)}")
print("=" * 60)

# Per-document table
if cleaning_reports:
    df = pd.DataFrame(cleaning_reports)[[
        "document_id", "source_file", "page_count", "block_count",
        "tables", "original_chars", "cleaned_chars", "reduction_pct",
        "warning_count", "needs_review"
    ]].copy()
    df["source_file"] = df["source_file"].apply(lambda x: Path(x).name)
    display(df)

# Save reports
with open(REPORT_FILE, 'w', encoding='utf-8') as f:
    json.dump({"summary": {
        "total_pdfs": total_pdfs,
        "succeeded":  total_success,
        "failed":     total_failed,
        "total_pages": total_pages,
        "total_blocks": total_blocks,
        "tables":     total_tables,
        "figures":    total_figs,
    }, "documents": cleaning_reports}, f, ensure_ascii=False, indent=2)

with open(FAILURE_FILE, 'w', encoding='utf-8') as f:
    json.dump(failure_reports, f, ensure_ascii=False, indent=2)

print(f"\n📄  Report saved → {REPORT_FILE}")
print(f"📄  Failures saved → {FAILURE_FILE}")


## 20. Preview Cleaned Documents

Displays representative examples of the cleaning transformations.


In [ ]:
# ── Preview: Raw → Cleaned (first 3 successful docs) ─────────────────────────
print("=" * 70)
print("PREVIEW: CLEANED DOCUMENT BLOCKS")
print("=" * 70)

if Path(JSONL_OUTPUT).exists():
    with open(JSONL_OUTPUT, encoding='utf-8') as f:
        preview_blocks = [json.loads(line) for line in f][:15]

    for b in preview_blocks:
        print(f"\n[{b['content_type'].upper():15s}] Page {b.get('page', '?')} | "
              f"Section: {b.get('section') or 'N/A'}")
        print(f"  {b['text'][:300]}{'...' if len(b['text']) > 300 else ''}")
        print(f"  Flags: {b.get('cleaning_flags') or 'none'}")
        print("-" * 70)
else:
    print("No JSONL output found — run the pipeline first.")


In [ ]:
# ── Preview: table conversion example ────────────────────────────────────────
print("=" * 70)
print("PREVIEW: TABLE CONVERSION EXAMPLE")
print("=" * 70)

# Demonstrate table_to_text on a synthetic example
demo_raw_table = [
    ["Medication", "Dose", "Frequency", "Route"],
    ["Ibuprofen",  "400 mg", "Every 6 hours", "Oral"],
    ["Acetaminophen", "500 mg", "Every 4–6 hours", "Oral"],
    ["Oxytocin", "10 IU", "As needed", "IV / IM"],
]

print("\nRAW TABLE (list of rows):")
for row in demo_raw_table:
    print("  |", " | ".join(row), "|")

demo_result = table_to_text(demo_raw_table, page_num=12, table_index=0)

print("\n→  CONVERTED TEXT REPRESENTATION:")
print(demo_result["text"])
print(f"\n   Flags: {demo_result['cleaning_flags'] or 'none'}")


In [ ]:
# ── Preview: hyphenation repair example ──────────────────────────────────────
print("=" * 70)
print("PREVIEW: HYPHENATION REPAIR EXAMPLE")
print("=" * 70)

demo_hyphens = """The patient should ensure ade-
quate hydration throughout the postpartum period.
Evidence-based guidelines recommend breast-
feeding within the first hour of delivery.
The follow-up appointment should be scheduled within six weeks.
Wound care must be performed daily to prevent infec-
tion and promote healing."""

print("\nRAW TEXT:")
print(demo_hyphens)

repaired, n_repairs = repair_hyphenation(demo_hyphens)
print(f"\n→  REPAIRED TEXT ({n_repairs} repairs):")
print(repaired)


## 21. Failure & Manual-Review Report


In [ ]:
print("=" * 60)
print("FAILURE REPORT")
print("=" * 60)

if failure_reports:
    for f in failure_reports:
        print(f"\n❌  {Path(f['source_file']).name}")
        print(f"    ID:    {f['document_id']}")
        print(f"    Error: {str(f['error'])[:400]}")
else:
    print("✅  No failures recorded.")

print()
print("=" * 60)
print("DOCUMENTS REQUIRING MANUAL REVIEW")
print("=" * 60)

if needs_review:
    for r in needs_review:
        print(f"\n⚠️  {Path(r['source_file']).name}")
        print(f"    Reduction: {r['reduction_pct']}%  (threshold: {REDUCTION_FLAG_THRESHOLD*100:.0f}%)")
        print(f"    Warnings:  {r['warning_count']}")
        print(f"    Blocks:    {r['block_count']}")
else:
    print("✅  All documents within acceptable cleaning thresholds.")

print(f"\n📌  Next steps:")
print(f"    1. Review flagged documents manually.")
print(f"    2. Inspect JSONL output: {JSONL_OUTPUT}")
print(f"    3. Proceed to Notebook 02 — Chunking & Embedding.")
